In [1]:
import json

file_path = "peptide_logits.json"
with open(file_path, "r") as file:
    data = json.load(file)
data["similarity_logits"]

[[13.750057220458984,
  1.8605175018310547,
  -4.151251316070557,
  -2.9003825187683105,
  -5.25892972946167,
  5.47551155090332,
  -5.435747146606445,
  0.6373155117034912,
  -2.7923085689544678,
  -3.267652988433838,
  3.4156172275543213,
  -0.8222441077232361,
  -3.250988006591797,
  -6.005999565124512,
  -1.5312753915786743,
  -1.2630589008331299,
  -1.1874040365219116,
  5.0873541831970215,
  -7.627072334289551,
  -5.483572006225586,
  -1.4012686014175415,
  -2.463711738586426,
  1.2725815773010254,
  -1.3319122791290283,
  -1.3441342115402222,
  -4.786804676055908,
  -4.127739906311035,
  -1.405243992805481,
  -1.0707197189331055,
  7.589687347412109,
  -0.1565907746553421,
  -2.2658779621124268,
  1.4531973600387573,
  2.1894631385803223,
  3.8861255645751953,
  1.6680889129638672,
  -4.36975622177124,
  3.920241594314575,
  1.1029753684997559,
  0.32274848222732544,
  0.21680350601673126,
  -1.1421340703964233,
  -5.234496593475342,
  -5.696269989013672,
  7.341752052307129,
  

In [2]:
import torch 
torch.tensor(data["similarity_logits"]).shape

torch.Size([256, 256])

In [6]:
def build_identity_matrix(values: list[str], device: torch.device) -> torch.Tensor:
        matrix = [[left == right for right in values] for left in values]
        return torch.tensor(matrix, dtype=torch.bool, device=device)
def _contrastive_logits_targets(
        similarity_logits: torch.Tensor,
        labels: torch.Tensor,
        smiles_list: list[str],
        sequence_list: list[str],
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, int]:
        device = similarity_logits.device
        batch_size = similarity_logits.shape[0]
        positive_mask = labels > 0.5
        positive_count = int(positive_mask.sum().item())

        if positive_count == 0:
            empty_logits = similarity_logits.new_empty((0, batch_size))
            empty_targets = torch.empty((0,), dtype=torch.long, device=device)
            return empty_logits, empty_targets, empty_logits, empty_targets, 0

        eye_mask = torch.eye(batch_size, dtype=torch.bool, device=device)
        smiles_match = build_identity_matrix(smiles_list, device)
        sequence_match = build_identity_matrix(sequence_list, device)
        positive_column_mask = positive_mask.unsqueeze(0)
        known_positive_mask = (smiles_match | sequence_match) & positive_column_mask
        invalid_negative_mask = known_positive_mask & ~eye_mask

        row_logits = similarity_logits[positive_mask].masked_fill(invalid_negative_mask[positive_mask], -1e9)
        row_targets = torch.arange(batch_size, device=device)[positive_mask]

        col_logits = similarity_logits.T[positive_mask].masked_fill(invalid_negative_mask.T[positive_mask], -1e9)
        col_targets = torch.arange(batch_size, device=device)[positive_mask]
        print(row_logits.shape)
        return row_logits, row_targets, col_logits, col_targets, positive_count
_contrastive_logits_targets(
        torch.tensor(data["similarity_logits"]),
        torch.tensor(data["labels"]),
        data["row_peptides"],
        data["column_sequences"],
    )

torch.Size([50, 256])


(tensor([[-3.1955, -3.0036,  0.4655,  ...,  0.2893,  4.7902, -6.1035],
         [-3.3640,  2.8061, -1.0701,  ...,  6.2175, -1.1675, -1.8640],
         [-0.3242,  6.2525, -0.0376,  ...,  3.7609,  7.9364, -5.4149],
         ...,
         [ 4.8304, -3.2970,  0.7475,  ..., -7.0383, -2.4766,  2.5914],
         [ 4.4225, -2.2300, -3.7602,  ..., -2.1794,  1.6300, -0.5540],
         [ 0.7610,  6.5637,  2.5601,  ..., -6.1080, -3.5692,  2.1301]]),
 tensor([  8,   9,  11,  19,  28,  33,  43,  45,  49,  52,  62,  63,  70,  78,
          80,  85,  87,  94,  95, 107, 118, 124, 125, 126, 127, 135, 144, 148,
         149, 159, 161, 163, 168, 169, 173, 181, 184, 187, 199, 203, 205, 210,
         212, 217, 219, 223, 228, 230, 234, 241]),
 tensor([[-2.7923e+00, -2.3357e+00,  1.0956e+00,  ..., -2.5361e-03,
           4.2457e+00, -6.8647e+00],
         [-3.2677e+00,  3.1184e+00, -8.9908e-01,  ...,  5.8102e+00,
          -4.3847e-01, -1.5831e+00],
         [-8.2224e-01,  6.7567e+00,  6.5408e-01,  ...,  3.50